# ECE-GY 9143 Lab 4 Part A: Distributed Deep Learning Demo

This Notebook is designed to run our DDP training experiments on interactive nodes like JupyterHub in the NYU HPC (Greene Cluster).

> **💡 Key Concept:**
Because PyTorch's `DistributedDataParallel (DDP)` requires spawning multiple independent processes to synchronize gradients, putting the main `train` loop directly inside a Notebook cell often leads to **Process Deadlocks** or **Out Of Memory (OOM)** errors. Therefore, the most stable and standard approach for distributed training is to keep the core logic in a `.py` script, and call it using `!python` or `%%bash` magics within the Notebook.

Please ensure that the compute node you are currently using has **at least 4 GPUs allocated (`--gres=gpu:4`)**.

## Experiment Q1: Computational Efficiency w.r.t Batch Size (Single GPU)

In [ ]:
!python lab4_ddp.py --run q1 --batch_size 32
!python lab4_ddp.py --run q1 --batch_size 128
!python lab4_ddp.py --run q1 --batch_size 512
!python lab4_ddp.py --run q1 --batch_size 2048

# Tip: You can keep doubling the batch_size (e.g., 4096, 8192) until the console 
# throws a `CUDA out of memory` error to find the max batch size your GPU can handle.

## Experiment Q2: Speedup Measurement (Multi-GPU DDP)

Here we will use 2 GPUs and 4 GPUs to verify how the training time decreases as we scale the number of GPUs.

In [ ]:
%%bash
echo "=== 2 GPUs Training ==="
torchrun --nproc_per_node=2 lab4_ddp.py --run q2 --batch_size 128

echo "=== 4 GPUs Training ==="
torchrun --nproc_per_node=4 lab4_ddp.py --run q2 --batch_size 128

## Experiment Q3: Computation VS Communication Overhead

After running this, compare the times with the single GPU training (same batch size) from Q1. You can estimate that `Training Time = Computation Time + AllReduce Communication Time`.

In [ ]:
%%bash
torchrun --nproc_per_node=2 lab4_ddp.py --run q3 --batch_size 128
torchrun --nproc_per_node=4 lab4_ddp.py --run q3 --batch_size 128

## Experiment Q4: Large Batch Training

We will advance the experiment to 5 epochs using the largest successful batch size tested in Q1 (e.g., 512 or larger). With 4 GPUs, our effective batch size will be `4 ✖️ batch_size`.

In [ ]:
%%bash
# Assuming the max available batch_size from your Q1 tests is 512
torchrun --nproc_per_node=4 lab4_ddp.py --run q4 --batch_size 512 --epochs 5